# Search — run a keyword search and save results to Postgres (daily job)

Runs the app's `search_task.py` from `src/app` (deployed with the bundle, so the job and the app share one copy of the code) on serverless. It calls the YouTube search API for `keyword`, pulls full video details + channels, and upserts them into Lakebase Postgres `public.videos` / `public.channels`, and appends discovery rows to `public.search_results`.

This is the headless equivalent of the “Tìm kiếm” button in the Streamlit app.

**Secrets (scope `youtube`):** `pg_password` (Postgres role `admin`), `api_key` (YouTube Data API v3).

Widget `dry_run` = `true` calls the API but writes nothing. `keyword`, `max_results`, `order` control the search.

In [ ]:
%pip install --quiet "psycopg[binary]==3.2.10" "httpx>=0.28.0" "python-dotenv>=1.0.0"


In [ ]:
# --- Config ---
import os
import sys

dbutils.widgets.text("keyword", "quán ăn Hồ Chí Minh")
dbutils.widgets.text("max_results", "10")
dbutils.widgets.dropdown("order", "relevance", ["relevance", "date", "viewCount"])
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"])

KEYWORD = dbutils.widgets.get("keyword")
MAX_RESULTS = int(dbutils.widgets.get("max_results"))
ORDER = dbutils.widgets.get("order")
DRY_RUN = dbutils.widgets.get("dry_run") == "true"

os.environ.update({
    "PGHOST": "ep-still-butterfly-d8fd3r70.database.us-east-2.cloud.databricks.com",
    "PGDATABASE": "databricks_postgres",
    "PGUSER": "admin",
    "PGPORT": "5432",
    "PGSSLMODE": "require",
    "APP_SCHEMA": "public",
    "PGPASSWORD": dbutils.secrets.get("youtube", "pg_password"),
    "YOUTUBE_API_KEY": dbutils.secrets.get("youtube", "api_key"),
})

notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
app_dir = os.path.normpath(f"/Workspace{notebook_dir}/../../app")  # src/notebooks/refresh -> src/app
sys.path.insert(0, app_dir)
print(f"App code: {app_dir} · keyword = {KEYWORD!r} · max_results = {MAX_RESULTS} · order = {ORDER} · dry_run = {DRY_RUN}")


In [ ]:
# --- Search + save (upserts videos/channels, appends search_results) ---
import search_task

counts = search_task.run_search(
    query=KEYWORD,
    max_results=MAX_RESULTS,
    order=ORDER,
    dry_run=DRY_RUN,
)
print(f"✅ Search complete: {counts}")
